# Juntar los archivos en una misma base de datos

In [1]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly_express as px
import numpy as np
import yfinance as yf
import pandas_market_calendars as mcal

In [2]:
dir_origen = '../DATOS/ORIGINALES'

dfs = [
    pd.read_csv(os.path.join(r, f))
    for r, _, files in os.walk(dir_origen)
    for f in files if f.lower().endswith('_marketstack.csv')
]

df = pd.concat(dfs, ignore_index=True)
df.to_csv("../DATOS/TRANSFORMADOS/dow_jones_consolidado.csv", index=False)

In [3]:
df["date"] = pd.to_datetime(df["date"])

In [4]:
df["date"] = pd.to_datetime(df["date"])
if df["date"].dt.tz is None:
    df["date"] = df["date"].dt.tz_localize("UTC")

# Convertir la columna de fecha a la hora local real de Nueva York
df["date"] = df["date"].dt.tz_convert("America/New_York")

# Filtrar días hábiles oficiales del mercado (NYSE)
nyse = mcal.get_calendar("NYSE")
schedule = nyse.schedule(
    start_date=df["date"].min().date(), end_date=df["date"].max().date()
)
dias_habiles_nyse = schedule.index.date

es_dia_laborable = df["date"].dt.date.isin(dias_habiles_nyse)

# Horario bolsa de 09:30 a 16:00
hora_apertura = pd.to_datetime("09:30").time()
hora_cierre = pd.to_datetime("16:00").time()

es_horario_mercado = (df["date"].dt.time >= hora_apertura) & (
    df["date"].dt.time <= hora_cierre
)

# Filtrar DataFrame manteniendo exclusivamente registros en sesión abierta
df_horario_mercado = df[es_dia_laborable & es_horario_mercado].copy()

# Remover zona horaria manteniendo la hora local de NY
df_horario_mercado["date"] = df_horario_mercado["date"].dt.tz_localize(None)
df_horario_mercado

,open,high,low,last,close,volume,date,symbol,exchange
3,213.133,213.730,212.985,NaN,213.480,NaN,2018-08-16 16:00:00,AAPL,IEXG
4,213.273,213.300,213.133,NaN,213.133,NaN,2018-08-16 15:30:00,AAPL,IEXG
5,212.785,213.465,212.785,NaN,213.133,NaN,2018-08-16 15:00:00,AAPL,IEXG
6,212.850,213.198,212.785,NaN,212.785,NaN,2018-08-16 14:30:00,AAPL,IEXG
7,211.780,213.198,211.780,NaN,212.785,NaN,2018-08-16 14:00:00,AAPL,IEXG
...,...,...,...,...,...,...,...,...,...
2744413,234.500,235.600,234.170,235.300,234.090,291094.0,2021-06-29 13:07:00,V,IEXG
2744414,234.500,235.600,234.170,235.235,234.090,290571.0,2021-06-29 13:05:00,V,IEXG
2744415,234.500,235.600,234.170,235.270,234.090,290254.0,2021-06-29 13:04:00,V,IEXG
2744416,234.500,235.600,234.170,235.255,234.090,289854.0,2021-06-29 13:03:00,V,IEXG


In [5]:
missings = df_horario_mercado.isna().sum()
print("Missings por columna:", missings)

Missings por columna: open             0
high             0
low              0
last        155101
close        78043
volume      152585
date             0
symbol           0
exchange         0
dtype: int64


In [6]:
# Agrupar días reales únicos por activo
df_horario_mercado["fecha_dia"] = df_horario_mercado["date"].dt.date
dias_reales = df_horario_mercado.groupby("symbol")["fecha_dia"].unique()

# Identificar días laborables faltantes entre min y max de cada activo
dias_faltantes = [
    {"symbol": activo, "fecha_faltante": dia.strftime("%Y-%m-%d")}
    for activo, fechas in dias_reales.items()
    for dia in set(pd.bdate_range(min(fechas), max(fechas)).date) - set(fechas)
]

df_dias_faltantes = pd.DataFrame(dias_faltantes).sort_values(
    ["symbol", "fecha_faltante"]
)
df_dias_faltantes

,symbol,fecha_faltante
9,AAPL,2018-07-04
6,AAPL,2018-09-03
5,AAPL,2018-11-22
24,AAPL,2018-12-05
25,AAPL,2018-12-25
...,...,...
389,V,2021-01-01
407,V,2021-01-18
391,V,2021-02-15
381,V,2021-04-02


In [7]:
print(df_horario_mercado.groupby("symbol")["date"].agg(["min", "max"]))

                       min                 max
symbol                                        
AAPL   2018-06-06 13:30:00 2021-06-30 16:00:00
AMGN   2020-05-29 11:35:00 2021-06-30 16:00:00
AXP    2020-05-29 11:30:59 2021-06-30 16:00:00
BA     2020-05-29 11:35:51 2021-06-30 16:00:00
CAT    2018-06-11 13:30:00 2021-06-30 16:00:00
CRM    2020-05-29 11:31:30 2021-06-30 16:00:00
CSCO   2020-05-29 11:31:26 2021-06-30 16:00:00
CVX    2020-05-29 11:31:23 2021-06-30 16:00:00
DIS    2020-05-29 11:35:00 2021-06-30 16:00:00
DOW    2020-05-29 11:35:56 2021-06-30 16:00:00
GS     2018-06-11 13:30:00 2021-06-30 16:00:00
HD     2020-05-29 13:55:51 2021-06-30 16:00:00
IBM    2020-05-29 13:56:29 2021-06-30 16:00:00
INTC   2018-06-07 14:00:00 2021-06-30 16:00:00
JNJ    2020-05-29 13:56:17 2021-06-30 16:00:00
JPM    2020-05-29 13:56:13 2021-06-30 16:00:00
KO     2020-05-29 13:56:55 2021-06-30 16:00:00
MCD    2020-05-29 13:54:54 2021-06-30 16:00:00
MMM    2020-05-29 13:54:34 2021-06-30 16:00:00
MRK    2020-0

In [8]:
# Agrupar por activo y día
jornadas = (
    df_horario_mercado.groupby(["symbol", df_horario_mercado["date"].dt.date])
    .agg(Apertura=("date", "min"), Cierre=("date", "max"))
    .reset_index()
)

# Filtrar aperturas distintos a 09:30 o cierres distintos a 16:00 y guardar en DataFrame
df_jornadas_irregulares = jornadas[
    (jornadas["Apertura"].dt.strftime("%H:%M") != "09:30")
    | (jornadas["Cierre"].dt.strftime("%H:%M") != "16:00")
].copy()

df_jornadas_irregulares["Fecha"] = df_jornadas_irregulares["Apertura"].dt.strftime("%Y-%m-%d")
df_jornadas_irregulares["Duracion_Horas"] = (
    (df_jornadas_irregulares["Cierre"] - df_jornadas_irregulares["Apertura"]).dt.total_seconds() / 3600
).round(2)
df_jornadas_irregulares["Apertura"] = df_jornadas_irregulares["Apertura"].dt.strftime("%H:%M")
df_jornadas_irregulares["Cierre"] = df_jornadas_irregulares["Cierre"].dt.strftime("%H:%M")

# Seleccionar y reordenar columnas finales
df_jornadas_irregulares = df_jornadas_irregulares[
    ["symbol", "Fecha", "Apertura", "Cierre", "Duracion_Horas"]
]
df_jornadas_irregulares

,symbol,Fecha,Apertura,Cierre,Duracion_Horas
0,AAPL,2018-06-06,13:30,16:00,2.50
1,AAPL,2018-06-07,13:30,16:00,2.50
2,AAPL,2018-06-08,13:30,16:00,2.50
3,AAPL,2018-06-11,13:30,16:00,2.50
4,AAPL,2018-06-12,13:30,16:00,2.50
...,...,...,...,...,...
10033,V,2021-05-27,09:31,16:00,6.48
10034,V,2021-05-28,09:31,16:00,6.48
10035,V,2021-06-01,09:31,16:00,6.48
10039,V,2021-06-07,09:31,16:00,6.48


In [9]:
# Calcular saltos dentro de la misma jornada
df_horario_mercado["gap"] = df_horario_mercado.groupby(
    ["symbol", df_horario_mercado["date"].dt.date]
)["date"].diff()

# Obtener el registro con el máximo salto por activo
res = df_horario_mercado.loc[
    df_horario_mercado.groupby("symbol")["gap"].idxmax()
].copy()

res["Desde_Hora"] = (res["date"] - res["gap"]).dt.strftime("%H:%M:%S")
res["Hasta_Hora"] = res["date"].dt.strftime("%H:%M:%S")
res["Fecha"] = res["date"].dt.strftime("%Y-%m-%d")
res["Minutos"] = (res["gap"].dt.total_seconds() / 60).round(1)
print("\nMAYOR SALTO INTRADÍA POR ACTIVO (DESDE / HASTA)")
print(
    res[["symbol", "Fecha", "Desde_Hora", "Hasta_Hora", "Minutos"]]
    .sort_values("Minutos", ascending=False)
    .to_string(index=False)
)


MAYOR SALTO INTRADÍA POR ACTIVO (DESDE / HASTA)
symbol      Fecha Desde_Hora Hasta_Hora  Minutos
  AAPL 2020-09-21   09:30:00   16:00:00    390.0
  AMGN 2021-03-05   09:30:00   16:00:00    390.0
   AXP 2020-10-23   09:30:00   16:00:00    390.0
    BA 2020-10-27   09:30:00   16:00:00    390.0
   CAT 2020-10-21   09:30:00   16:00:00    390.0
   CRM 2020-10-13   09:30:00   16:00:00    390.0
  CSCO 2020-06-16   09:30:00   16:00:00    390.0
   CVX 2020-10-22   09:30:00   16:00:00    390.0
   DIS 2020-10-23   09:30:00   16:00:00    390.0
   DOW 2020-10-22   09:30:00   16:00:00    390.0
    GS 2020-10-22   09:30:00   16:00:00    390.0
    HD 2020-10-22   09:30:00   16:00:00    390.0
   IBM 2020-10-23   09:30:00   16:00:00    390.0
  INTC 2020-10-26   09:30:00   16:00:00    390.0
   JNJ 2020-10-23   09:30:00   16:00:00    390.0
   JPM 2021-03-15   09:30:00   16:00:00    390.0
    KO 2021-03-03   09:30:00   16:00:00    390.0
   MCD 2020-06-16   09:30:00   16:00:00    390.0
   MMM 2020-10-22   

In [10]:
# Porcentaje de nulos en close por activo
df_nulos_close = (
    df_horario_mercado.groupby("symbol")["close"]
    .apply(lambda x: (x.isnull().sum() / len(x)) * 100)
    .round(2)
    .reset_index(name="pct_nulos_close")
    .sort_values("pct_nulos_close", ascending=False)
)

print(df_nulos_close.to_string(index=False))

symbol  pct_nulos_close
   TRV             6.39
  AMGN             4.89
   AXP             4.86
    BA             4.27
   MMM             4.21
   DOW             4.21
  CSCO             4.09
   CRM             3.99
   CAT             3.98
  MSFT             3.97
   IBM             3.94
   CVX             3.89
   DIS             3.86
   UNH             3.83
   JPM             3.65
   MCD             3.64
    HD             3.61
   JNJ             3.55
    GS             3.51
     V             3.49
  INTC             3.44
    KO             3.30
    PG             3.30
   MRK             3.26
  AAPL             3.08
   NKE             3.00


In [11]:
# fig = px.line(
#     df_horario_mercado,
#     x="date",
#     y="close",
#     color="symbol",
#     title="Evolución del Precio de Cierre (Close) por Activo - Detección de Splits",
#     labels={"date": "Fecha", "close": "Precio de Cierre (USD)", "activo": "Ticker"},
# )

# fig.update_layout(
#     xaxis_title="Fecha",
#     yaxis_title="Precio de Cierre ($)",
#     hovermode="x unified",
#     legend_title_text="Activo",
#     height=650,
#     template="plotly_white"
# )

# fig.show()

In [12]:
# Recuperar Días Completos
if not df_dias_faltantes.empty:
    tickers = df_dias_faltantes["symbol"].unique().tolist()
    f_min = df_dias_faltantes["fecha_faltante"].min()
    f_max = (
        pd.to_datetime(df_dias_faltantes["fecha_faltante"].max())
        + pd.Timedelta(days=1)
    ).strftime("%Y-%m-%d")

    df_yf_dias = (
        yf.download(
            tickers, start=f_min, end=f_max, interval="1d", group_by="ticker"
        )
        .stack(level=0, future_stack=True)
        .reset_index()
    )
    df_yf_dias = df_yf_dias.rename(
        columns={
            "Date": "date",
            "Ticker": "symbol",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Volume": "volume",
        }
    )
    df_yf_dias["date"] = pd.to_datetime(df_yf_dias["date"])

    # Filtrar solo las fechas que faltaban
    df_dias_rec = pd.merge(
        df_dias_faltantes,
        df_yf_dias,
        left_on=["symbol", "fecha_faltante"],
        right_on=["symbol", df_yf_dias["date"].dt.strftime("%Y-%m-%d")],
    ).drop(columns=["fecha_faltante"])
else:
    df_dias_rec = pd.DataFrame()

# Unificar y limpiar en el DataFrame final
cols = ["symbol", "date", "open", "high", "low", "close", "volume"]
df_consolidado_completo = (
    pd.concat([df_horario_mercado[cols], df_dias_rec[cols]], ignore_index=True)
    .drop_duplicates(subset=["symbol", "date"])
    .sort_values(["symbol", "date"])
    .reset_index(drop=True)
)

[*********************100%***********************]  26 of 26 completed


In [13]:
# Unificar formato de fecha
df_consolidado_completo["date"] = pd.to_datetime(
    df_consolidado_completo["date"]
).dt.tz_localize(None)
df_consolidado_completo["fecha_dia"] = df_consolidado_completo["date"].dt.date

# Días laborables faltantes
nyse = mcal.get_calendar("NYSE")
faltantes = [
    {"symbol": symbol, "fecha_faltante": dia.strftime("%Y-%m-%d")}
    for symbol, g in df_consolidado_completo.groupby("symbol")
    for dia in set(
        nyse.schedule(
            start_date=g["fecha_dia"].min(), end_date=g["fecha_dia"].max()
        ).index.date
    )
    - set(g["fecha_dia"].unique())
]
df_dias_faltantes_post = pd.DataFrame(faltantes)

# Jornadas irregulares
jornadas = (
    df_consolidado_completo.groupby(["symbol", "fecha_dia"])
    .agg(
        apertura=("date", lambda x: x.min().strftime("%H:%M")),
        cierre=("date", lambda x: x.max().strftime("%H:%M")),
    )
    .reset_index()
)

df_jornadas_irregulares_post = jornadas[
    (jornadas["apertura"] != "09:30") | (jornadas["cierre"] != "16:00")
].reset_index(drop=True)

# Resumen
print(f"Días faltantes restantes: {len(df_dias_faltantes_post)}")
print(f"Jornadas irregulares restantes: {len(df_jornadas_irregulares_post)}")

Días faltantes restantes: 0
Jornadas irregulares restantes: 8371


In [14]:
df_pivot = df_consolidado_completo.pivot(index='date', columns='symbol').swaplevel(axis=1).sort_index(axis=1)
columnas_validas = ["open", "high", "low", "close", "volume"]
df_pivot = df_pivot.loc[:, (slice(None), columnas_validas)]

In [15]:
agg_dict = {}
for col in df_pivot.columns:
    ticker, metric = col
    
    if metric == 'open':
        agg_dict[col] = 'first' 
    elif metric == 'high':
        agg_dict[col] = 'max'  
    elif metric == 'low':
        agg_dict[col] = 'min'  
    elif metric == 'close':
        agg_dict[col] = 'last' 
    elif metric == 'volume':
        agg_dict[col] = 'sum'  

df_daily = df_pivot.groupby(df_pivot.index.normalize()).agg(agg_dict)

df_daily = df_daily.dropna(how='all')

In [16]:
tickers = df['symbol'].unique().tolist()

datos_yf = yf.download(tickers, start="2018-06-06", end="2021-06-30", auto_adjust=False, actions=True)

splits_data = datos_yf['Stock Splits']

tickers_with_splits = splits_data.loc[:, (splits_data > 0).any(axis=0)]

actual_splits = tickers_with_splits[(tickers_with_splits > 0).any(axis=1)]

print(actual_splits.dropna(how='all'))

[*********************100%***********************]  26 of 26 completed

Ticker      AAPL    MRK
Date                   
2020-08-31   4.0  0.000
2021-06-03   0.0  1.048


In [17]:
# Ajustar Close a Splits
prices_to_adjust = ['open', 'high', 'low', 'close']

for split_date, row in actual_splits.iterrows():
    split_date_str = split_date.strftime('%Y-%m-%d')
    
    for ticker, ratio in row.items():
        if ratio > 0.0:
            print(f"Applying {ratio}-for-1 split adjustment for {ticker} prior to and including {split_date_str}")
            
            pre_split_mask = df_daily.index.normalize() <= split_date_str
            
            for col in prices_to_adjust:
                df_daily.loc[pre_split_mask, (ticker, col)] /= ratio
            
            df_daily.loc[pre_split_mask, (ticker, 'volume')] *= ratio

print("\nAll historical splits applied successfully.")

Applying 4.0-for-1 split adjustment for AAPL prior to and including 2020-08-31
Applying 1.048-for-1 split adjustment for MRK prior to and including 2021-06-03

All historical splits applied successfully.


In [18]:
# # Choose the tickers you want to inspect in subplots
# raw_tickers  = df_consolidado_completo.columns.get_level_values(0).unique()
# tickers_to_plot = [tk for tk in raw_tickers if isinstance(tk, str) and tk.isupper() and 1 <= len(tk) <= 5]

# # 2. Create subplots (stacked vertically, sharing the X-axis for easy timeline comparison)
# fig = make_subplots(
#     rows=len(tickers_to_plot), 
#     cols=1, 
#     shared_xaxes=True, 
#     subplot_titles=tickers_to_plot
# )

# # 3. Add a line trace for each ticker into its own subplot row
# for i, tk in enumerate(tickers_to_plot, start=1):
#     ticker_close = df_consolidado_completo[tk]['close'].dropna()
    
#     fig.add_trace(
#         go.Scatter(
#             x=ticker_close.index, 
#             y=ticker_close.values, 
#             mode='lines', 
#             name=tk,
#             line=dict(width=1.5)
#         ),
#         row=i, 
#         col=1
#     )

# # 4. Update layout for a clean appearance
# fig.update_layout(
#     height=300 * len(tickers_to_plot), # Dynamic height based on number of subplots
#     title_text="Historical Close Prices (Interactive Subplots)",
#     template="plotly_white",
#     showlegend=False
# )

# # 5. Render directly in Jupyter safely
# fig.show()

In [19]:
# Calendario y rangos
f_ini, f_fin = "2018-06-06", "2025-06-06"
dias_nyse = set(mcal.get_calendar("NYSE").schedule(start_date=f_ini, end_date=f_fin).index.date)

# Asegurar que el índice base es datetime sin timezone
df_daily.index = pd.to_datetime(df_daily.index).tz_localize(None)

# Obtener min/max de fechas válidas para CADA símbolo en el df_daily
r = {}
for tk in df_daily.columns.get_level_values(0).unique():
    # Eliminar NaNs para encontrar el rango real de datos que tenemos para este símbolo
    valid_dates = df_daily[tk].dropna(how='all').index
    if not valid_dates.empty:
        r[tk] = {'min': valid_dates.min(), 'max': valid_dates.max()}

# Descargar tramos faltantes
dfs_to_add = []
cols = ["open", "high", "low", "close", "volume"]

def get_yf_multi(tk, s, e):
    try:
        # Descargamos los datos faltantes (usamos auto_adjust=True para manejar splits/dividendos recientes)
        d = yf.download(tk, start=s, end=e, interval="1d", progress=False, auto_adjust=True)
        
        if d.empty:
            return
            
        # Manejo de compatibilidad con diferentes versiones de yfinance
        if isinstance(d.columns, pd.MultiIndex):
            d.columns = d.columns.get_level_values(0)
            
        # Normalizar nombres de columnas a minúsculas
        d.columns = [c.lower() for c in d.columns]
        d = d[cols] # Mantener solo OHLCV
        
        # Normalizar fecha en el índice
        d.index = pd.to_datetime(d.index)
        d.index = (
            d.index.tz_localize(None) if d.index.tz is None 
            else d.index.tz_convert("America/New_York").tz_localize(None)
        )
        
        # Convertir a MultiIndex para que encaje perfectamente con df_pivot
        # Nivel 0: Símbolo (tk), Nivel 1: Atributos (open, high, etc.)
        d.columns = pd.MultiIndex.from_product([[tk], d.columns])
        
        dfs_to_add.append(d)
    except Exception as e:
        print(f"Error descargando {tk}: {e}")

# Revisar qué fechas faltan e importarlas
for tk, dates in r.items():
    min_str = dates["min"].strftime("%Y-%m-%d")
    max_str = dates["max"].strftime("%Y-%m-%d")
    
    if min_str > f_ini:
        print(f"Descargando {tk} desde {f_ini} hasta {min_str}")
        get_yf_multi(tk, f_ini, min_str)
        
    if max_str < f_fin:
        print(f"Descargando {tk} desde {max_str} hasta {f_fin}")
        get_yf_multi(tk, max_str, f_fin)

# Limpiar y consolidar todo en un solo DataFrame
if dfs_to_add:
    # Unir todos los fragmentos nuevos de yfinance verticalmente
    new_data = pd.concat(dfs_to_add, axis=0)
    
    # Agrupar por fecha para fusionar las filas y asegurar índice único
    new_data = new_data.groupby(new_data.index).first()
    
    # Asegurar que df_pivot no tenga fechas duplicadas
    df_daily = df_daily[~df_daily.index.duplicated(keep='last')]
    
    # Ordenar las columnas (MultiIndex) alfabéticamente en ambos DataFrames 
    df_daily = df_daily.sort_index(axis=1)
    new_data = new_data.sort_index(axis=1)

    columnas_validas = ["open", "high", "low", "close", "volume"]
    df_daily = df_daily.loc[:, (slice(None), columnas_validas)]

# Igualar los nombres de los niveles (Pandas es muy estricto con esto)
    df_daily.columns.names = ['Ticker', 'Atributo']
    new_data.columns.names = ['Ticker', 'Atributo']

# Ordenar alfabéticamente para que los índices encajen a la perfección
    df_daily = df_daily.sort_index(axis=1)
    new_data = new_data.sort_index(axis=1)

# Imprimir para confirmar visualmente antes de la fusión
    print("Estructura df_daily:", type(df_daily.columns))
    print("Estructura new_data:", type(new_data.columns))
    
    df_consolidado_completo = df_daily.combine_first(new_data)

# Filtrar días válidos de NYSE
df_consolidado_completo = df_consolidado_completo[pd.Index(df_consolidado_completo.index.date).isin(dias_nyse)]

print("\n--- Rangos finales en el DataFrame Consolidado ---")
for tk in df_consolidado_completo.columns.get_level_values(0).unique():
    fechas = df_consolidado_completo[tk].dropna(how='all').index
    if not fechas.empty:
        print(f"{tk}: Min {fechas.min().strftime('%Y-%m-%d')} | Max {fechas.max().strftime('%Y-%m-%d')}")

Descargando AAPL desde 2021-06-30 hasta 2025-06-06
Descargando AMGN desde 2021-06-30 hasta 2025-06-06
Descargando AXP desde 2021-06-30 hasta 2025-06-06
Descargando BA desde 2021-06-30 hasta 2025-06-06
Descargando CAT desde 2021-06-30 hasta 2025-06-06
Descargando CRM desde 2021-06-30 hasta 2025-06-06
Descargando CSCO desde 2021-06-30 hasta 2025-06-06
Descargando CVX desde 2021-06-30 hasta 2025-06-06
Descargando DIS desde 2021-06-30 hasta 2025-06-06
Descargando DOW desde 2021-06-30 hasta 2025-06-06
Descargando GS desde 2021-06-30 hasta 2025-06-06
Descargando HD desde 2021-06-30 hasta 2025-06-06
Descargando IBM desde 2021-06-30 hasta 2025-06-06
Descargando INTC desde 2021-06-30 hasta 2025-06-06
Descargando JNJ desde 2021-06-30 hasta 2025-06-06
Descargando JPM desde 2021-06-30 hasta 2025-06-06
Descargando KO desde 2021-06-30 hasta 2025-06-06
Descargando MCD desde 2021-06-30 hasta 2025-06-06
Descargando MMM desde 2021-06-30 hasta 2025-06-06
Descargando MRK desde 2021-06-30 hasta 2025-06-06


In [19]:
df_consolidado_completo

,symbol,date,open,high,low,close,volume,fecha_dia
0,AAPL,2018-06-06 13:30:00,193.630,193.920,192.96,193.000,NaN,2018-06-06
1,AAPL,2018-06-06 14:00:00,192.980,193.040,191.95,192.405,NaN,2018-06-06
2,AAPL,2018-06-06 14:30:00,192.720,192.765,191.95,192.405,NaN,2018-06-06
3,AAPL,2018-06-06 15:00:00,192.440,193.260,192.43,193.070,NaN,2018-06-06
4,AAPL,2018-06-06 15:30:00,193.170,193.260,192.77,193.070,NaN,2018-06-06
...,...,...,...,...,...,...,...,...
2027424,V,2021-06-30 15:56:00,235.505,236.380,233.88,235.950,197449.0,2021-06-30
2027425,V,2021-06-30 15:57:00,235.505,236.380,233.88,235.950,201290.0,2021-06-30
2027426,V,2021-06-30 15:58:00,234.880,234.940,234.88,NaN,NaN,2021-06-30
2027427,V,2021-06-30 15:59:00,235.505,236.380,233.88,235.950,207842.0,2021-06-30


In [21]:
# Ajusta splits y dividendos (reducir a ajustar solamente los splits, no dividendos)

tickers = df_consolidado_completo.columns.get_level_values(0).unique().tolist()
tickers = [str(t).strip() for t in tickers] 

datos_yf = yf.download(tickers, start="2018-06-06", end="2025-06-06", auto_adjust=False, actions=True, progress=False)

div_data = datos_yf['Dividends']
close_data = datos_yf['Close']

actual_divs = div_data.loc[:, (div_data > 0).any(axis=0)]
actual_divs = actual_divs[(actual_divs > 0).any(axis=1)]

prices_to_adjust = ['open', 'high', 'low', 'close']

for div_date, row in actual_divs.iterrows():
    div_date_str = div_date.strftime('%Y-%m-%d')
    for ticker, div_amount in row.items():
        if pd.notna(div_amount) and div_amount > 0.0:
            
            prior_closes = close_data[ticker].loc[close_data.index < div_date_str]
            
            if not prior_closes.empty:
                prev_close = prior_closes.iloc[-1]
                if pd.notna(prev_close) and prev_close > 0:
                    adj_ratio = 1 - (div_amount / prev_close)
                    
                    pre_div_mask = df_consolidado_completo.index.normalize() < div_date_str
                    for col in prices_to_adjust:
                        df_consolidado_completo.loc[pre_div_mask, (ticker, col)] *= adj_ratio

$DATE: possibly delisted; no timezone found
$VOLUME: possibly delisted; no timezone found
$CLOSE: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: FECHA_DIA"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: SYMBOL"}}}
$FECHA_DIA: possibly delisted; no timezone found
$SYMBOL: possibly delisted; no timezone found

5 Failed downloads:
['DATE', 'VOLUME', 'CLOSE', 'FECHA_DIA', 'SYMBOL']: possibly delisted; no timezone found


AttributeError: 'RangeIndex' object has no attribute 'normalize'

In [20]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Load your daily parquet data
df = pd.read_parquet("../DATOS/TRANSFORMADOS/df_diario.parquet")

# Choose the tickers you want to inspect in subplots
raw_tickers = df.columns.get_level_values(0).unique()
tickers_to_plot = [tk for tk in raw_tickers if isinstance(tk, str) and tk.isupper() and 1 <= len(tk) <= 5]

# 2. Create subplots (stacked vertically, sharing the X-axis for easy timeline comparison)
fig = make_subplots(
    rows=len(tickers_to_plot), 
    cols=1, 
    shared_xaxes=True, 
    subplot_titles=tickers_to_plot
)

# 3. Add a line trace for each ticker into its own subplot row
for i, tk in enumerate(tickers_to_plot, start=1):
    ticker_close = df[tk]['close'].dropna()
    
    fig.add_trace(
        go.Scatter(
            x=ticker_close.index, 
            y=ticker_close.values, 
            mode='lines', 
            name=tk,
            line=dict(width=1.5)
        ),
        row=i, 
        col=1
    )

# 4. Update layout for a clean appearance
fig.update_layout(
    height=300 * len(tickers_to_plot), # Dynamic height based on number of subplots
    title_text="Historical Close Prices (Interactive Subplots)",
    template="plotly_white",
    showlegend=False
)

# 5. Render directly in Jupyter safely
fig.show()

In [22]:
window = 5
threshold = 0.15 

for tk in df_consolidado_completo.columns.get_level_values(0).unique():
    if (tk, 'close') in df_consolidado_completo.columns:
        close_series = df_consolidado_completo[tk]['close']
        
        rolling_med = close_series.rolling(window=window, center=True).median()
        
        deviation = np.abs((close_series - rolling_med) / rolling_med)
        
        df_consolidado_completo.loc[deviation > threshold, (tk, 'close')] = np.nan
        df_consolidado_completo[(tk, 'close')] = df_consolidado_completo[(tk, 'close')].interpolate(method='linear')

In [28]:
# 1. Forward fill to cover random missing mid-week days
df_consolidado_completo = df_consolidado_completo.ffill()

# 2. Drop rows where the entire market was closed (weekends/holidays)
df_consolidado_completo = df_consolidado_completo.dropna(how='all')

# 3. Check what is left (usually just pre-IPO dates, which are fine)
remaining_nas = df_consolidado_completo.isna().sum()
print("Remaining NAs per column:\n", remaining_nas[remaining_nas > 0])

Remaining NAs per column:
 Ticker  Atributo
AMGN    close       498
        high        498
        low         498
        open        498
AXP     close       498
                   ... 
UNH     open        500
V       close         3
        high          3
        low           3
        open          3
Length: 100, dtype: int64


In [23]:
df_daily.to_parquet("../DATOS/TRANSFORMADOS/df_diario.parquet")